# Salecta: run from a browser (no local setup)

Use this when you don't have your own computer set up. It needs **no GPU**. Click **Runtime → Run all**
(Colab) or **Run All** (Kaggle, with *Internet: On*), paste your OpenRouter key when asked, and download
the zip at the end.

Pick what to run in the second cell: the full Salecta run (3 models, 180 prompts each), or grading
of refusal-direction results.

In [ ]:
!git clone -q https://github.com/Lord-Shax/Salecta.git 2>/dev/null || (cd Salecta && git pull -q)
%cd Salecta
!pip -q install -e ".[dev,providers]"
import os
from getpass import getpass
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key, then press Enter: ").strip()
print("Key length:", len(os.environ["OPENROUTER_API_KEY"]), "(should be 73)")

## A. Full Salecta run (3 models)
Takes roughly 10–20 minutes. Safe to re-run: finished models are skipped.

In [ ]:
!python scripts/full_run.py

In [ ]:
# Download the results (Colab). On Kaggle, find salecta_full_run.zip in the Output panel instead.
try:
    from google.colab import files
    files.download("salecta_full_run.zip")
except ImportError:
    import shutil; shutil.copy("salecta_full_run.zip", "/kaggle/working/") if os.path.isdir("/kaggle/working") else None
    print("Saved salecta_full_run.zip")

## B. Grade refusal-direction results (optional)
Upload your `results_<model>.zip` files first (Colab: folder icon → upload, into the `Salecta` folder),
then run this cell. It writes one `graded_<model>.csv` per zip.

In [ ]:
import glob, subprocess, sys
zips = sorted(glob.glob("results_*.zip") + glob.glob("../results_*.zip") + glob.glob("/kaggle/input/**/results_*.zip", recursive=True))
print("Found:", zips or "no results_*.zip files yet")
for z in zips:
    local = os.path.basename(z)
    if os.path.abspath(z) != os.path.abspath(local):
        import shutil; shutil.copy(z, local)
    subprocess.run([sys.executable, "scripts/grade_replies.py", local], check=False)
print("Graded files:", sorted(glob.glob("graded_*.csv")))